# sponsor-scout: train and score on Colab

This notebook trains the LoRA adapter from the repo and scores it on the 300 synthetic **dev** adverts. It does not touch the test split.

**Before you start**
- Runtime > Change runtime type > a GPU. Free Colab usually gives a T4 (16 GB).
- The base model download is about 7.6 GB. Training time on the author's 8 GB RTX 4070 was 54 minutes. A T4 will be slower; no Colab time has been measured yet (**TBC**).
- A T4 has no native bfloat16, so the code falls back to float16. That path is **untested until this notebook has been run once**.
- Keep the browser tab open. A disconnected session loses the adapter.
- Results will not match the repo's numbers exactly (GPU maths is not bit-identical). For reference, the author's run scored 0.994 mean field accuracy on dev.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > pick a GPU"
print(torch.__version__, torch.cuda.get_device_name(0))
print("native bf16:", torch.cuda.is_bf16_supported(including_emulation=False))

In [ ]:
!git clone https://github.com/Onkar-Gupta/sponsor-scout
%cd sponsor-scout
!pip install -q -r requirements.txt
!pip install -q -e .
!pip install -q -r requirements-train.txt
import os
os.environ["HF_HOME"] = "/content/hf"   # keeps the model download in Colab's disk

## 1. Smoke test (5 steps)
This confirms the model loads and trains on this GPU before the long run. It uses none of the training budget.

In [ ]:
!python scripts/train.py --name check --rank 16 --lr 0.0002 --epochs 1 --smoke

## 2. Train (rank 16, learning rate 2e-4, 2 epochs, 2,000 adverts)
These are the settings the repo's protocol froze. The run name must be new, because the repo already holds a run called `r16_lr2e-4_e2`.

In [ ]:
!python scripts/train.py --name colab_r16_lr2e-4_e2 --rank 16 --lr 0.0002 --epochs 2

## 3. Run the tuned model on the dev adverts and score it

In [ ]:
!python scripts/run_arm.py --arm tuned --adapter outputs/adapters/colab_r16_lr2e-4_e2 --split dev --batch-size 8 --out outputs/colab_dev_tuned.jsonl
!python scripts/score_run.py --gold data/dev.jsonl --pred outputs/colab_dev_tuned.jsonl --out reports/colab_dev_score.json > /dev/null
import json
r = json.load(open("reports/colab_dev_score.json"))["all"]
print({k: round(r[k], 3) for k in ("json_validity", "mean_field_accuracy", "sponsorship_accuracy", "sponsorship_macro_f1")})

## 4. Try your own advert
Paste an advert between the quotes and run the cell.

In [ ]:
advert = '''
Data Analyst
Brightco Ltd
Location: Leeds
Salary: GBP 40,000 - 50,000
We are unable to offer sponsorship.
'''
open("advert.txt", "w").write(advert)
!sponsor-scout extract --file advert.txt --arm tuned --adapter outputs/adapters/colab_r16_lr2e-4_e2